<a href="https://colab.research.google.com/github/dacode100/Sistemas-operativos/blob/colab-sistemas/Daniel_Ramirez_Practica2_Conceptos_SO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica1_Conceptos_SO.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.

# 💻 Sistemas Operativos - Exploración de la Estructura del SO y Recursos del Sistema
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad I:** Conceptos y Evolución de los Sistemas Operativos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Objetivos de Aprendizaje
1. Identificar la estructura física y lógica de la máquina virtual Linux subyacente en Google Colab.
2. Analizar el rol del Kernel en la administración de recursos hardware (CPU, Memoria, E/S).
3. Ejecutar y comprender llamadas al sistema (Syscalls) y la arquitectura en modo Kernel/Usuario.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede a tu entorno de **NotebookLM del Curso**, abre la sección de la *Unidad I* y consulta:
---

- *¿Qué es el Kernel de Linux y cómo abstrae el hardware?*  
**RESPUESTA:** El Kernel de Linux es la parte central del sistema operativo Linux que se ejecuta en modo kernel (modo privilegiado o supervisor), interactuando directamente con el hardware y administrando los recursos del computador para las aplicaciones y servicios del sistema. Aunque tiene una arquitectura monolítica (al igual que la mayoría de los sistemas tipo UNIX), admite módulos cargables que permiten añadir o quitar funcionalidades —como controladores de dispositivos, sistemas de archivos o protocolos de red— de forma dinámica en tiempo de ejecución.   1. ¿Qué es y qué hace el Kernel de Linux?El kernel actúa como intermediario y gestor de recursos entre el hardware físico y los programas que corren en modo usuario. Sus responsabilidades principales incluyen:   Administración de procesos y planificación: Crea, gestiona y planifica tareas (representadas internamente mediante task_struct, sin distinguir rígidamente entre procesos e hilos), maneja señales y realiza cambios de contexto.   Administración de memoria: Gestiona la memoria física y la memoria virtual (tablas de páginas, paginación bajo demanda, caché de páginas, políticas de reemplazo y asignación dinámica como el algoritmo buddy para módulos).   Gestión de entrada/salida (E/S) y sistemas de archivos: Coordina la comunicación con dispositivos de bloque, carácter y red, y unifica el acceso a archivos y directorios (por ejemplo, mediante el Sistema de archivos virtuales o VFS).   Control de interrupciones, excepciones y llamadas al sistema: Atiende interrupciones hardware y trampas (traps), y ofrece la interfaz de llamadas al sistema (system calls) para que los programas en modo usuario soliciten servicios protegidos.   2. ¿Cómo abstrae el hardware?A diferencia de diseños como Windows NT, que emplean una capa dedicada explícita llamada HAL (Hardware Abstraction Layer) para aislar ciertas diferencias de plataforma, el kernel de Linux abstrae el hardware mediante una combinación de interfaces estandarizadas, subsistemas en capas y controladores (drivers) que ocultan las particularidades físicas detrás de modelos lógicos uniformes:   Llamadas al sistema y bibliotecas estándar: En lugar de que las aplicaciones manipulen puertos de E/S, registros de hardware o direcciones físicas directamente, solicitan servicios mediante llamadas al sistema (y bibliotecas como la estándar de C / POSIX), provocando un cambio seguro de modo usuario a modo kernel.   Controladores de dispositivos (drivers) y código independiente del hardware: Cada tipo de dispositivo cuenta con un controlador que aísla al resto del sistema de las peculiaridades del hardware físico, dividiendo su trabajo entre la lógica genérica/independiente del dispositivo y las rutinas específicas que programan el controlador del hardware.   Clasificación uniforme de E/S y VFS: Linux agrupa los controladores en categorías claras —principalmente dispositivos de bloques (con acceso aleatorio, como discos) y dispositivos de carácter (flujo de bytes, como terminales o impresoras), además del manejo de red mediante sockets— y los integra bajo abstracciones como el Sistema de archivos virtuales (VFS), de modo que leer de un archivo o de un dispositivo se trate de forma uniforme en los niveles superiores.   Memoria virtual y "procesador virtual": Mediante paginación y la MMU, el kernel ofrece a cada proceso un espacio de direcciones virtuales propio y protegido en lugar de exponer la memoria física cruda, mientras el planificador y los mecanismos de multitarea hacen que cada tarea disponga de una visión ordenada de ejecución sobre la CPU.   Abstracción de interrupciones y concurrencia: Oculta a los programas de usuario las interrupciones del hardware (que dependen de la arquitectura) y las gestiona internamente —por ejemplo, separando el trabajo urgente (mitad superior) del menos urgente (mitad inferior) y soportando multiprocesamiento simétrico (SMP)— para presentar un entorno estable y portable entre distintas arquitecturas.  

- *¿Qué función cumple la jerarquía de memoria en un SO moderno?*  
**RESPUESTA:**  En un sistema operativo moderno, la jerarquía de memoria cumple la función de resolver el compromiso entre velocidad, capacidad y costo de las distintas tecnologías de almacenamiento, permitiendo que el procesador acceda a la información a alta velocidad mientras se dispone de gran capacidad a un costo razonable. Como ninguna memoria física individual es a la vez ultrarrápida, enorme, no volátil y barata, el sistema organiza el almacenamiento en niveles y abstrae y administra el movimiento de datos entre ellos para ofrecer a los programas un modelo de memoria amplio, cómodo y eficiente.   1. Principio de diseño: compensación y localidadA medida que se desciende por los niveles de la jerarquía (desde los registros y la caché hasta la memoria principal y el almacenamiento secundario como discos o cintas), ocurre lo siguiente:Capacidad y costo: Aumenta la capacidad total de almacenamiento y disminuye el costo por bit.   Velocidad y frecuencia de acceso: Aumenta el tiempo de acceso (se vuelve más lenta), pero disminuye la frecuencia con la que el procesador necesita bajar a esos niveles inferiores.   La clave para que el sistema rinda cerca de la velocidad de los niveles superiores radica en el principio de localidad (o cercanía de referencias): durante una ventana de tiempo, los programas tienden a referenciar repetidamente un subconjunto acotado de instrucciones y datos (localidad temporal) y direcciones cercanas entre sí (localidad espacial).   2. Función del sistema operativo en la jerarquíaAunque el hardware gestiona directamente ciertos niveles (como las cachés de procesador mediante controladores/hardware especializado y la traducción de direcciones con la MMU), el sistema operativo cumple funciones esenciales sobre la jerarquía:   Abstracción y memoria virtual: Oculta las limitaciones físicas y la complejidad de los niveles inferiores mediante esquemas como la memoria virtual (apoyada en paginación o segmentación), dando a cada proceso la ilusión de un espacio de direcciones propio, grande y protegido, y moviendo bloques (páginas o segmentos) bajo demanda entre memoria principal y almacenamiento secundario (swap o dispositivo de paginación).   Migración, reemplazo y coherencia: Decide qué información subir a niveles más rápidos, qué porciones expulsar cuando el nivel superior se llena (evitando la hiperpaginación o thrashing cuando los conjuntos residentes son insuficientes) y cómo mantener la coherencia entre copias en distintos niveles —por ejemplo, mediante cachés de disco/páginas y políticas de escritura retrasada o agrupada—.   Maximización del rendimiento y la multiprogramación: Al mantener en memoria principal únicamente las partes activas o necesarias de los procesos, permite alojar más procesos simultáneamente, mantener ocupada la CPU y reducir el impacto de las operaciones de E/S lentas.   Aislamiento y compartición controlada: Administra la asignación y liberación de memoria, protege los espacios de cada proceso y del kernel, y habilita mecanismos seguros como la memoria compartida o el afinamiento de caché/NUMA.   3. Niveles típicos en la jerarquíaNivel habitualCaracterísticas y rolGestión predominanteRegistrosMáxima velocidad y menor tamaño; integran el procesador.   Compilador / Hardware   Memoria cachéRápida y costosa; intermedia entre CPU y memoria principal para datos/instrucciones recientes.   Hardware (y coherencia/software en ciertos casos)   Memoria principal (RAM)Volátil y de capacidad intermedia; aloja programas y datos en uso y actúa como respaldo/caché de disco.   Sistema operativo (con apoyo de MMU)   Almacenamiento secundario (disco, etc.)Gran capacidad, menor costo y no volátil; guarda información permanente y sirve de respaldo a la memoria virtual.   Sistema operativo   

---

## 3. Sección Práctica 1: Inspección de Recursos del Sistema (Bash / CLI)

En esta sección exploraremos el sistema sobre el cual se ejecuta este Cuaderno (Servidor Linux provisto por Google).

### [CÓDIGO 1.1] Identificación del Kernel y la Distribución Linux
```bash
# Ejecutamos comandos Bash usando el prefijo !
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

In [ ]:
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

=== INFORMACIÓN DEL KERNEL ===
Linux 34fe20f254cf 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux

=== DISTRIBUCIÓN DE LINUX ===
PRETTY_NAME="Ubuntu 22.04.5 LTS"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy


### [CÓDIGO 1.2] Inspección de la CPU (Recurso Procesador)
```bash
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

In [ ]:
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

=== ARQUITECTURA Y NÚCLEOS DE CPU ===
Architecture:                            x86_64
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Core(s) per socket:                      1
NUMA node0 CPU(s):                       0,1


### [CÓDIGO 1.3] Inspección de la Memoria RAM y Jerarquía de Almacenamiento
```bash
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURA ==="
!df -h /

In [ ]:
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURA ==="
!df -h /

=== ESTADO DE LA MEMORIA RAM (en MB) ===
               total        used        free      shared  buff/cache   available
Mem:           12975         786        8955           2        3232       11906
Swap:              0           0           0

=== SISTEMA DE ARCHIVOS Y DISCO DURA ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /


### Pregunta de Análisis 1 (Responde en la celda de texto):
Basándote en las lecturas de Tanenbaum en NotebookLM y en los resultados obtenidos arriba:

1. ¿Qué versión de Kernel Linux está ejecutando la máquina virtual de Colab?  
**RESPUESTA:** 6.6.122+

2. ¿Cuántos núcleos de procesador y cuánta memoria RAM tiene asignados tu entorno virtual?  
**RESPUESTA:** Nucleos del procesador: 2  
 memoria ram: 12875 MB

## 4. Sección Práctica 2: Abstracción de Hardware a través del Sistema `/proc`

En Linux, `/proc` es un **pseudo-sistema de archivos**. No ocupa espacio en disco; es una interfaz directa mediante la cual el Kernel expone información sobre el estado de los recursos y procesos del sistema.

### [CÓDIGO 2.1] Exploración del sistema /proc
```bash
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

In [ ]:
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ===
MemTotal:       13286936 kB
MemFree:         9136396 kB
MemAvailable:   12168720 kB
Buffers:          132904 kB
Cached:          3096872 kB
SwapCached:            0 kB
Active:           479992 kB
Inactive:        3422784 kB
Active(anon):       1572 kB
Inactive(anon):   673764 kB

=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ===
1086.62 1824.71


### [CÓDIGO 2.2] Visualización del árbol de llamadas e interrupciones del hardware
```bash
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

In [ ]:
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

=== INTERRUPCIONES DE HARDWARE REGISTRADAS ===
           CPU0       CPU1       
  0:        262          0   IO-APIC   0-edge      timer
  1:          0          9   IO-APIC   1-edge      i8042
  4:          0       2776   IO-APIC   4-edge      ttyS0
  8:          0          0   IO-APIC   8-edge      rtc0
  9:          0          0   IO-APIC   9-fasteoi   acpi
 10:       1215          0   IO-APIC  10-fasteoi   virtio2
 12:          3          0   IO-APIC  12-edge      i8042
 24:          0          0  PCI-MSIX-0000:00:06.0   0-edge      virtio3-config
 25:          0         32  PCI-MSIX-0000:00:06.0   1-edge      virtio3-input
 26:          0          0  PCI-MSIX-0000:00:03.0   0-edge      virtio0-config
 27:          0          0  PCI-MSIX-0000:00:03.0   1-edge      virtio0-control
 28:          0          0  PCI-MSIX-0000:00:03.0   2-edge      virtio0-event
 29:          0     130373  PCI-MSIX-0000:00:03.0   3-edge      virtio0-request
 30:          0          0  PCI-MSIX-0000:00:0

## 5. Sección Práctica 3: Primer Contacto con Bajo Nivel (C / Syscalls POSIX)

A continuación, crearemos, compilaremos y ejecutaremos un programa en C para observar directamente las **Llamadas al Sistema** (`sys_write`, `sys_getpid`) que interactúan entre el **Modo Usuario** y el **Modo Kernel**.

```bash
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

In [ ]:
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

# include <stdio.h>
# include <unistd.h>
# include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

Writing syscall_demo.c


### [CÓDIGO 3.2] Compilación y Ejecución del Código C usando GCC
```bash
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

In [ ]:
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

--------------------------------------------------
[MODO USUARIO] Ejecutando programa en C...
[LLAMADA AL SISTEMA] PID del Proceso Actual: 6205
[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 1579
--------------------------------------------------


### [CÓDIGO 3.3] Rastreando las llamadas al sistema reales con `strace`
```bash
# Instalamos strace para interceptar las llamadas que hace el ejecutable al Kernel
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

In [ ]:
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

strace: Can't stat './syscall_demo': No such file or directory


## 6. Desafío Asíncrono / Entregable de la Unidad I

### Instrucciones:
1. Modifica el archivo `syscall_demo.c` de la Sección 5 creando un nuevo archivo `desafio_u1.c` mediante celdas mágicas `%%writefile`.
2. Agrega la llamada al sistema POSIX `getuid()` (para obtener el ID del usuario que ejecuta el proceso) y la función `sleep(5)` para pausar la ejecución durante 5 segundos.
3. Ejecuta el programa en segundo plano mediante Bash y utiliza el comando `ps aux` para capturar el proceso en la lista de procesos activos mientras duerme.
4. Vuelve a ejecutar `strace` sobre el nuevo binario para identificar las syscalls `nanosleep` y `getuid`.

---

In [ ]:
%%writefile desafio_u1.c
#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    pid_t process_id = getpid();
    pid_t parent_id = getppid();
    uid_t user_id = getuid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre: %d\n", parent_id);
    printf("[LLAMADA AL SISTEMA] UID del Usuario: %d\n", user_id);
    printf("--------------------------------------------------\n");

    sleep(5);

    return 0;
}

Overwriting desafio_u1.c


### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica1_Conceptos_SO.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.